# Geography Vector — `recipe_geography_vectors.npy`

**Mission**: reproduce the recipe **geography vector** of the ICOCO paper from the new
`ICOCO.duckdb` pipeline (legacy reference: `old_code/10_geo_vector/geo_vector_npy_export.ipynb`).

**Output**: `/workspace/7_create_npy_files/recipe_geography_vectors.npy`,
dtype `float32`, shape `(50511, 249)` — one row per recipe (`silver.RecipeZenodo` rowid order),
one column per country (`silver.countries` rowid order = CSV order of `coutries.csv`).

**Inputs**

| Input | Where |
|---|---|
| Recipe ingredient occurrences | `silver.RecipeZenodo.Ingredients` (`VARCHAR[]`, ingredient names only) |
| Dirty ingredient → GBIF country list | `bronze.dirty_ingredient_country_mapping` (`input`, `country_list VARCHAR[]`) |
| Country codes (fixed column order) | `silver.countries` (`Name`, `Code`) |

**Legacy deltas (expected, documented)**: the legacy artifact was `(50513, 248)` — the old
silver CSV had 2 rows that the new bronze ingestion dropped, and the legacy `coutries.csv`
had 248 distinct codes vs 249 now. The 2 row-drop positions (old rows **21475 / 23805**)
were located by two-pointer alignment on the odor/entity matrices and are used **only** for
the legacy comparison in section 5, never for building the vector.

## 0. Setup — connect → close (flush WAL) → reconnect

Same connection pattern as `silver_curation.ipynb` / `explore_zenodo_recipe.ipynb`:
close first so the WAL file (`ICOCO.duckdb.wal`) is flushed, then reconnect fresh.
This notebook only **reads** DuckDB and writes **TEMP** objects (session-local, vanish on
disconnect), so re-running it never changes the persistent database state.

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100  # show all rows in query results
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL)
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

In [4]:
from sql.connection import ConnectionManager

# IMPORTANT: DuckDB allows only ONE connection to the database at a time.
# Reuse the SAME connection that %sql already opened (do NOT duckdb.connect() again).
raw = ConnectionManager().current.connection_sqlalchemy.connection

# Keep RAM safe inside the 3.68 GB container: cap DuckDB memory + threads.
raw.execute("SET memory_limit = '1.5GB'")
raw.execute("SET threads = 4")
print('Reusing the single %sql connection for Python-side fetches (raw):', raw)

Reusing the single %sql connection for Python-side fetches (raw): <sqlalchemy.pool.base._ConnectionFairy object at 0x739a94909ea0>


## 1. Sanity checks

Row counts of the three inputs before building anything.

In [5]:
%%sql
SELECT 'silver.RecipeZenodo' AS tbl, count(*) AS rows FROM silver.RecipeZenodo
UNION ALL SELECT 'silver.countries', count(*) FROM silver.countries
UNION ALL SELECT 'bronze.dirty_ingredient_country_mapping', count(*) FROM bronze.dirty_ingredient_country_mapping;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

tbl,rows
silver.RecipeZenodo,50511
silver.countries,249
bronze.dirty_ingredient_country_mapping,4280


## 2. Build the long-format vector (pure SQL)

`CREATE OR REPLACE TEMP VIEW` keeps this idempotent; the view dies with the session.

In [6]:
%%sql
-- Long-format geography vector entries.
-- Faithful to the legacy Python logic (old_code/10_geo_vector/geo_vector_npy_export.ipynb):
--   * ingredient occurrences are counted per recipe (duplicates kept);
--   * the denominator counts EVERY country occurrence returned by the mapping,
--     even codes missing from silver.countries (they simply get no column);
--   * share = country count / total country occurrences of the recipe.
CREATE OR REPLACE TEMP VIEW geo_long AS
WITH ing AS (
    -- one row per ingredient OCCURRENCE; rowid = silver row order (0-based)
    SELECT r.rowid AS recipe_idx, lower(trim(u.ing)) AS ing
    FROM silver.RecipeZenodo r, UNNEST(r.Ingredients) AS u(ing)
),
matched AS (
    SELECT i.recipe_idx, UNNEST(m.country_list) AS country
    FROM ing i
    JOIN bronze.dirty_ingredient_country_mapping m
      ON lower(trim(m.input)) = i.ing
),
tally AS (
    SELECT recipe_idx, country, COUNT(*) AS cnt,
           SUM(COUNT(*)) OVER (PARTITION BY recipe_idx) AS total
    FROM matched
    GROUP BY recipe_idx, country
)
SELECT t.recipe_idx,
       c.rowid AS country_idx,          -- countries keep their CSV order (rowid)
       c.Code  AS country_code,
       t.cnt,
       t.total,
       t.cnt::DOUBLE / t.total AS share
FROM tally t
JOIN silver.countries c ON c.Code = t.country;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [7]:
%%sql
SELECT * FROM geo_long LIMIT 8;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

recipe_idx,country_idx,country_code,cnt,total,share
3,121,LA,1,4,0.25
3,30,BR,1,4,0.25
3,197,SC,1,4,0.25
3,48,KM,1,4,0.25
6,165,NO,1,6,0.16666666666666666
6,47,CO,1,6,0.16666666666666666
6,216,TW,1,6,0.16666666666666666
6,92,GN,1,6,0.16666666666666666


## 3. Coverage statistics

In [8]:
%%sql
SELECT count(*)                  AS long_rows,
       count(DISTINCT recipe_idx)  AS recipes_with_geo_signal,
       count(DISTINCT country_idx) AS countries_actually_used,
       sum(cnt)                    AS total_country_occurrences
FROM geo_long;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

long_rows,recipes_with_geo_signal,countries_actually_used,total_country_occurrences
281161,34408,109,311866


## 4. Pivot to a dense matrix and save `.npy`

Python is only thin glue: fetch the small long-format result over the **same** connection
and scatter it into a dense `float32` matrix.

In [9]:
import numpy as np
import os

OUT = '/workspace/7_create_npy_files/recipe_geography_vectors.npy'

df = raw.execute('SELECT recipe_idx, country_idx, share FROM geo_long').df()
n_recipes   = raw.execute('SELECT count(*) FROM silver.RecipeZenodo').fetchone()[0]
n_countries = raw.execute('SELECT count(*) FROM silver.countries').fetchone()[0]

geo = np.zeros((n_recipes, n_countries), dtype=np.float32)
geo[df['recipe_idx'].to_numpy(np.int64), df['country_idx'].to_numpy(np.int64)] = df['share'].to_numpy(np.float32)

np.save(OUT, geo)
print('saved:', OUT, '| shape:', geo.shape, '| dtype:', geo.dtype,
      '| size:', round(os.path.getsize(OUT) / 1e6, 2), 'MB')

saved: /workspace/7_create_npy_files/recipe_geography_vectors.npy | shape: (50511, 249) | dtype: float32 | size: 50.31 MB


## 5. Verification

**(a) internal** — per-row shares must sum to ≤ 1 (== 1 whenever the recipe has at least one
matched ingredient) plus a spot-check of a known recipe.

**(b) vs the legacy artifact** — compare the "has geography signal" pattern row-by-row with
`old_code/10_geo_vector/data/recipe_geography_vectors.npy` under the documented 2-row drop
alignment. Country *columns* are also compared on the pre-shift region `0..150` (the legacy
country list diverges by +1 column after old index ~151 because it held one code fewer).

In [10]:
import numpy as np

# ---- (a) internal ----
row_sums = geo.sum(axis=1)
print('rows total:', geo.shape[0])
print('rows with signal (sum > 0):', int((row_sums > 0).sum()))
print('max row sum (must be <= 1 + 1e-5):', round(float(row_sums.max()), 6))

name, ings = raw.execute(
    "SELECT Name, Ingredients FROM silver.RecipeZenodo WHERE rowid = 0").fetchone()
r0 = geo[0]
print(f'spot-check rowid 0 {name!r}: ingredients = {ings}')
print('  nonzero country shares:', {int(i): round(float(r0[i]), 4) for i in np.nonzero(r0)[0]})

rows total: 50511
rows with signal (sum > 0): 34408
max row sum (must be <= 1 + 1e-5): 1.0
spot-check rowid 0 'Pineapple Glaze for Ham': ingredients = ['pineapple', 'maraschino cherries', 'brown sugar']
  nonzero country shares: {}


In [11]:
import numpy as np

# ---- (b) vs legacy artifact (read-only, mmap -> RAM friendly) ----
OLD = '/workspace/old_code/10_geo_vector/data/recipe_geography_vectors.npy'
old = np.load(OLD, mmap_mode='r')
print('legacy shape:', old.shape, '| new shape:', geo.shape)

old_idx = np.empty(geo.shape[0], dtype=np.int64)
_o = 0
for _j in range(geo.shape[0]):
    while _o in (21475, 23805):   # old rows dropped by the new bronze ingestion
        _o += 1
    old_idx[_j] = _o
    _o += 1
old_rows = np.asarray(old[old_idx], dtype=np.float32)

sig_old = old_rows.sum(axis=1) > 0.5
sig_new = geo.sum(axis=1) > 0.5
print('recipes with geo signal — legacy:', int(sig_old.sum()), '| new:', int(sig_new.sum()))
print('signal agreement:', round(float((sig_old == sig_new).mean()) * 100, 3), '%')

d = np.abs(old_rows[:, :151] - geo[:, :151])
print('pre-shift country columns 0..150: max|diff| =', float(d.max()),
      '| rows differing > 1e-6:', int((d.max(axis=1) > 1e-6).sum()))

legacy shape: (50513, 248) | new shape: (50511, 249)


recipes with geo signal — legacy: 34408 | new: 34408
signal agreement: 100.0 %
pre-shift country columns 0..150: max|diff| = 0.08888889104127884 | rows differing > 1e-6: 223


## 6. Disconnect

Close the jupysql connection; the `TEMP` views die with the session, leaving the
database file untouched.

In [12]:
%sql --close duckdb:////workspace/ICOCO.duckdb